# 08. DiLoCo Local SGD PowerSGD and Distributed Checkpointing

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Douillard et al. (DiLoCo, Google DeepMind 2023, *DiLoCo: Distributed Low-Communication Training of Language Models*), Vogels et al. (PowerSGD, NeurIPS 2019), PyTorch Distributed Team (Distributed Checkpointing, 2024).
- **Purpose:** Overcome global interconnect bottlenecks via bilevel local SGD (DiLoCo), low-rank gradient compression (PowerSGD), and non-blocking sharded checkpoint resharding.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Expert Parallelism and 3D 4D Parallel Composition](07_expert_parallelism_and_3d_4d_parallel_composition.ipynb) $\to$ `🔗 Downstream Syntheses:` [Deployment Low Level and Capstones](../../README.md#17_deployment_low_level_and_capstones)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use DiLoCo when high-bandwidth intra-cluster interconnects (e.g. InfiniBand within a single datacenter) are available. In tightly-coupled clusters, synchronous AllReduce/FSDP2 achieves strictly faster wall-clock convergence without the outer optimization tuning hyperparameter overhead.

## Mathematical Foundations & Governing Equations

$$
\bar{\Delta} = \frac{1}{W} \sum_{w=1}^W (\theta_0 - \theta_H^{(w)}), \quad v_{t+1} = \beta v_t + \bar{\Delta}, \quad \theta_{t+1} = \theta_0 - \eta_{\text{out}} v_{t+1}, \quad \hat{G} = P Q^\top
$$

### Asymptotic Time & Memory Complexity
- **Communication Reduction:** DiLoCo cuts network synchronization frequency by $H\times$ (typically $H=500$), reducing WAN bandwidth demand by $99.8\%$.
- **Gradient Compression:** PowerSGD reduces gradient transfer volume from $O(M \cdot N)$ to $O((M+N)r)$.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def diloco_outer_step(theta_0: torch.Tensor, theta_workers: torch.Tensor, v_outer: torch.Tensor, lr_outer: float = 0.7, beta_outer: float = 0.9) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    deltas = theta_0.unsqueeze(0) - theta_workers
    delta_avg = torch.mean(deltas, dim=0)
    v_new = beta_outer * v_outer + delta_avg
    theta_new = theta_0 - lr_outer * v_new
    return theta_new, v_new, delta_avg

def powersgd_compress_decompress(grad_matrix: torch.Tensor, probe_q: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    p = grad_matrix @ probe_q
    p_norm = p / (torch.norm(p, p=2) + 1e-8)
    q_new = grad_matrix.t() @ p_norm
    return p_norm @ q_new.t(), p_norm, q_new

def distributed_checkpoint_reshard(shards_2: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    full_tensor = torch.cat([shards_2[0], shards_2[1]], dim=0)
    return full_tensor, torch.stack(torch.chunk(full_tensor, 4, dim=0), dim=0)

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

theta_new, v_new, delta_avg = diloco_outer_step(weights["theta_0"], weights["theta_workers"], weights["v_outer_init"], 0.7, 0.9)
assert float(torch.max(torch.abs(theta_new - weights["theta_new"]))) < 1e-5
assert float(torch.max(torch.abs(v_new - weights["v_outer_new"]))) < 1e-5
assert float(torch.max(torch.abs(delta_avg - weights["delta_avg"]))) < 1e-5

recon, p_norm, q_new = powersgd_compress_decompress(weights["grad_matrix"], weights["probe_q"])
assert float(torch.max(torch.abs(recon - weights["powersgd_recon"]))) < 1e-5
assert float(torch.max(torch.abs(p_norm - weights["powersgd_p"]))) < 1e-5
assert float(torch.max(torch.abs(q_new - weights["powersgd_q"]))) < 1e-5

full_tensor, shards_4 = distributed_checkpoint_reshard(weights["shards_2"])
assert float(torch.max(torch.abs(shards_4 - weights["shards_4"]))) < 1e-5
print("All DiLoCo, PowerSGD, and DCP Resharding parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: diloco_outer_step(weights["theta_0"], weights["theta_workers"], weights["v_outer_init"]), iters=50, name="DiLoCo Outer Step"),
    benchmark_fn(lambda: powersgd_compress_decompress(weights["grad_matrix"], weights["probe_q"]), iters=50, name="PowerSGD Compress"),
    benchmark_fn(lambda: distributed_checkpoint_reshard(weights["shards_2"]), iters=50, name="DCP Resharding"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Synchronous AllReduce over WAN vs DiLoCo Outer Sync
num_params = 70_000_000_000  # 70B parameter model
h_local_steps = 500
bytes_per_param = 2  # FP16 gradients

# Per-step DDP AllReduce syncs parameters every single step:
sync_per_step_gb = (num_params * bytes_per_param * 2) / (1024 ** 3)  # 2x ring overhead = 280 GB / step
total_ddp_wan_tb = (sync_per_step_gb * h_local_steps) / 1024

# DiLoCo syncs only once every H steps:
total_diloco_wan_tb = sync_per_step_gb / 1024

print(f"Synchronous DDP WAN Volume ({h_local_steps} steps): {total_ddp_wan_tb:.2f} TB")
print(f"DiLoCo Outer Loop WAN Volume:           {total_diloco_wan_tb:.2f} TB  ({total_ddp_wan_tb / total_diloco_wan_tb:.1f}x bandwidth reduction)")
assert total_ddp_wan_tb == h_local_steps * total_diloco_wan_tb, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Bilevel Decoupling:** Separating fast local model updates from infrequent outer trajectory reconciliation reduces communication volume by $H\times$.
2. **Low-Rank Gradient Squeezing:** PowerSGD preserves gradient directions via alternating power iterations with error-feedback residual memory buffers.
3. **Zero-Copy Checkpoint Resharding:** Recording coordinate bounding boxes enables transparently saving on $N_1$ nodes and loading onto $N_2$ nodes.

---
### Cross-Domain Generalisation Matrix
| Domain | Low-Frequency Synchronization Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Federated Edge Computing** | Local mobile model fine-tuning with periodic central aggregation | Preserves user data privacy while saving mobile battery and cellular data |
| **Git Distributed Version Control** | Local commit branches merged periodically to remote upstream | Eliminates central server network bottleneck during offline code development |
| **Distributed Ledger & Blockchain** | Epoch-based block consensus validation | Bounds cross-validator gossip traffic across global peer-to-peer nodes |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [08_diloco_local_sgd_powersgd_and_distributed_checkpointing.md](../../markdown/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.md)
- **PyTorch 2.x Module:** [08_diloco_local_sgd_powersgd_and_distributed_checkpointing.py](../../pytorch2/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.py)
- **Functional JAX Module:** [08_diloco_local_sgd_powersgd_and_distributed_checkpointing.py](../../jax/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.py)
- **Native Rust Candle Track:** [08_diloco_local_sgd_powersgd_and_distributed_checkpointing.rs](../../candle/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.rs)
- **Hardware Kernel (CPU / WGSL):** [08_diloco_local_sgd_powersgd_and_distributed_checkpointing.rs](../../kernels/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.rs)
- **Golden SafeTensors:** [08_diloco_local_sgd_powersgd_and_distributed_checkpointing.safetensors](../../weights/16_distributed_and_scalable/08_diloco_local_sgd_powersgd_and_distributed_checkpointing.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Expert Parallelism and 3D 4D Parallel Composition](07_expert_parallelism_and_3d_4d_parallel_composition.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 17. Deployment Low Level and Capstones](../../README.md#17_deployment_low_level_and_capstones) &rarr;